# Advanced Problems with Solutions: Iterating Callables

This notebook focuses on the second form of Python's `iter()`:

```python
iter(callable, sentinel)
```

It repeatedly calls a zero-argument callable and yields each returned value until the returned value compares equal to the sentinel.

The source lesson introduces callable-based iterators with closures, a custom `CounterIterator`, sentinel-based stopping, random values, and countdown examples.


## Learning goals

By the end of this notebook, you should be able to:

- Build stateful callables using closures and classes.
- Use `iter(callable, sentinel)` safely and idiomatically.
- Explain why the sentinel value is consumed but not yielded.
- Handle exhaustion correctly.
- Avoid infinite loops with guardrails.
- Use callable iterators for files, polling, pagination, random streams, and simulated APIs.
- Understand equality-based sentinel matching and common edge cases.


## Best-practice checklist

1. Prefer `iter(callable, sentinel)` over writing a custom iterator when the logic is "call until this value appears".
2. Remember that the callable must accept **no arguments**.
3. Remember that the sentinel is tested using equality (`==`), not identity (`is`).
4. Choose a sentinel that cannot be confused with valid data.
5. Add a maximum-iteration guard when a sentinel may never appear.
6. Avoid using mutable sentinels unless the behavior is intentional and tested.
7. Keep state local by using closures or callable objects.
8. Do not hide unexpected exceptions; adapt only the exceptions you intentionally want to treat as "end of stream".


## Setup: reusable helpers


In [1]:
from collections import deque
from functools import partial
from io import StringIO
from itertools import count, islice, takewhile
import random
import time


## Warm-up example: a stateful counter closure

A closure can store private state and expose a zero-argument function.


In [2]:
def make_counter(start=0, step=1):
    current = start

    def next_value():
        nonlocal current
        current += step
        return current

    return next_value


counter = make_counter(start=0, step=1)
assert counter() == 1
assert counter() == 2
assert counter() == 3


## Warm-up example: built-in callable iterator

`iter(counter, 5)` repeatedly calls `counter()` and stops when the returned value equals `5`.

Important: `5` is **not yielded**.


In [3]:
counter = make_counter(start=0, step=1)
values = list(iter(counter, 5))

assert values == [1, 2, 3, 4]


## Problem 1 — Rebuild `iter(callable, sentinel)` manually

Implement a class named `CallableSentinelIterator`.

Requirements:

- Accept a zero-argument callable and a sentinel.
- Yield values returned by the callable.
- Stop when the returned value equals the sentinel.
- After it stops once, it must remain exhausted forever.
- Raise `TypeError` if the first argument is not callable.


In [4]:
class CallableSentinelIterator:
    def __init__(self, func, sentinel):
        if not callable(func):
            raise TypeError("func must be callable")
        self._func = func
        self._sentinel = sentinel
        self._exhausted = False

    def __iter__(self):
        return self

    def __next__(self):
        if self._exhausted:
            raise StopIteration

        value = self._func()

        if value == self._sentinel:
            self._exhausted = True
            raise StopIteration

        return value


counter = make_counter()
it = CallableSentinelIterator(counter, 4)

assert list(it) == [1, 2, 3]
assert list(it) == []          # remains exhausted
assert iter(it) is it          # iterator protocol

try:
    CallableSentinelIterator(42, None)
except TypeError as ex:
    assert "callable" in str(ex)
else:
    raise AssertionError("Expected TypeError")


## Problem 2 — Compare the custom class to built-in `iter`

Use the custom `CallableSentinelIterator` and built-in `iter(callable, sentinel)` on equivalent counters.

Confirm that both produce the same values.


In [5]:
custom_counter = make_counter(start=10, step=2)
builtin_counter = make_counter(start=10, step=2)

custom_values = list(CallableSentinelIterator(custom_counter, 20))
builtin_values = list(iter(builtin_counter, 20))

assert custom_values == [12, 14, 16, 18]
assert builtin_values == [12, 14, 16, 18]
assert custom_values == builtin_values


## Problem 3 — Prove the sentinel is consumed but not yielded

Create a counter that returns `1, 2, 3, 4, 5, ...`.

Use `iter(callable, 3)` and prove that `3` is not in the result.


In [6]:
counter = make_counter()
values = list(iter(counter, 3))

assert values == [1, 2]
assert 3 not in values


## Problem 4 — Confirm that the built-in callable iterator remains exhausted

Once `iter(callable, sentinel)` sees its sentinel, it should not keep calling the callable.


In [7]:
calls = []

def source():
    calls.append("called")
    return len(calls)

it = iter(source, 3)

assert next(it) == 1
assert next(it) == 2

try:
    next(it)
except StopIteration:
    pass
else:
    raise AssertionError("Expected StopIteration when sentinel is reached")

calls_after_stop = len(calls)

for _ in range(5):
    try:
        next(it)
    except StopIteration:
        pass
    else:
        raise AssertionError("Iterator should remain exhausted")

assert len(calls) == calls_after_stop


## Problem 5 — Build a configurable countdown closure

Create `make_countdown(start, stop)` that returns a zero-argument callable.

Each call should decrease the current number by `1` and return the new value.

Use `iter(callable, stop - 1)` so that the `stop` value is yielded.


In [8]:
def make_countdown(start, stop=0):
    current = start

    def next_value():
        nonlocal current
        current -= 1
        return current

    return next_value


takeoff = make_countdown(start=5, stop=0)
values = list(iter(takeoff, -1))

assert values == [4, 3, 2, 1, 0]


## Problem 6 — Countdown with validation

Improve the countdown factory so invalid inputs raise a helpful error.

Requirements:

- `start` must be greater than `stop`.
- Both values must be integers.


In [9]:
def make_validated_countdown(start, stop=0):
    if not isinstance(start, int) or not isinstance(stop, int):
        raise TypeError("start and stop must be integers")
    if start <= stop:
        raise ValueError("start must be greater than stop")

    current = start

    def next_value():
        nonlocal current
        current -= 1
        return current

    return next_value


takeoff = make_validated_countdown(4, 0)
assert list(iter(takeoff, -1)) == [3, 2, 1, 0]

for args, expected_error in [((0, 0), ValueError), ((1.5, 0), TypeError)]:
    try:
        make_validated_countdown(*args)
    except expected_error:
        pass
    else:
        raise AssertionError(f"Expected {expected_error.__name__} for {args}")


## Problem 7 — Use `iter(callable, sentinel)` with seeded random numbers

Create a local random number generator using `random.Random(seed)` so the code is deterministic and does not affect global random state.

Generate random integers from `0` to `10` until the value `8` appears.


In [10]:
rng = random.Random(0)
random_until_8 = iter(lambda: rng.randint(0, 10), 8)

values = list(random_until_8)

assert values == [6, 6, 0, 4]
assert 8 not in values


## Problem 8 — Add a maximum-call guard

Sometimes the sentinel may never appear. Build a wrapper that returns the sentinel after a maximum number of calls.

This prevents accidental infinite loops.


In [11]:
def with_max_calls(func, max_calls, sentinel):
    if max_calls < 0:
        raise ValueError("max_calls must be non-negative")

    calls = 0

    def guarded():
        nonlocal calls
        if calls >= max_calls:
            return sentinel
        calls += 1
        return func()

    return guarded


rng = random.Random(123)
SENTINEL = object()

guarded_random = with_max_calls(lambda: rng.randint(1, 10), max_calls=5, sentinel=SENTINEL)
values = list(iter(guarded_random, SENTINEL))

assert len(values) == 5
assert all(1 <= value <= 10 for value in values)


## Problem 9 — Read fixed-size chunks from a file-like object

A classic use of `iter(callable, sentinel)` is reading from a file until an empty string or empty bytes object is returned.

Use `StringIO` to simulate a file.


In [12]:
stream = StringIO("abcdefghij")

chunks = list(iter(lambda: stream.read(3), ""))

assert chunks == ["abc", "def", "ghi", "j"]


## Problem 10 — Use `functools.partial` instead of `lambda`

`iter(callable, sentinel)` needs a zero-argument callable.

When the real function needs arguments, `partial` can pre-fill them.


In [13]:
stream = StringIO("PythonIterators")

read_four = partial(stream.read, 4)
chunks = list(iter(read_four, ""))

assert chunks == ["Pyth", "onIt", "erat", "ors"]


## Problem 11 — Simulate reading binary blocks

For binary streams, the sentinel is usually `b""`, not `""`.


In [14]:
class BytesReader:
    def __init__(self, data, block_size):
        self._data = data
        self._block_size = block_size
        self._index = 0

    def read(self):
        if self._index >= len(self._data):
            return b""

        block = self._data[self._index:self._index + self._block_size]
        self._index += self._block_size
        return block


reader = BytesReader(b"abcdefg", block_size=2)
blocks = list(iter(reader.read, b""))

assert blocks == [b"ab", b"cd", b"ef", b"g"]


## Problem 12 — Adapt an exception-based source

Some APIs signal the end by raising an exception rather than returning a sentinel.

Write an adapter that converts a specific exception into a sentinel value.

Best practice: catch only the exception you expect.


In [15]:
class EndOfStream(Exception):
    pass


def make_exception_source(values):
    items = deque(values)

    def next_item():
        if not items:
            raise EndOfStream
        return items.popleft()

    return next_item


def end_as_sentinel(func, end_exception, sentinel):
    def adapted():
        try:
            return func()
        except end_exception:
            return sentinel

    return adapted


END = object()
source = make_exception_source(["A", "B", "C"])
adapted_source = end_as_sentinel(source, EndOfStream, END)

values = list(iter(adapted_source, END))

assert values == ["A", "B", "C"]


## Problem 13 — Understand equality-based sentinel matching

The sentinel is matched using equality (`==`), not identity (`is`).

This can surprise you when objects define unusual equality behavior.


In [16]:
class MatchesEverything:
    def __eq__(self, other):
        return True


data = iter([1, 2, 3, 4])
sentinel = MatchesEverything()

# Because 1 == sentinel eventually returns True, iteration stops immediately.
values = list(iter(data.__next__, sentinel))

assert values == []


## Problem 14 — Prefer a unique sentinel object when valid data may include `None`

If `None` can be real data, do not use `None` as the sentinel.

Use a unique object instead.


In [17]:
END = object()
items = deque([None, "alpha", None, "omega", END])

values = list(iter(items.popleft, END))

assert values == [None, "alpha", None, "omega"]


## Problem 15 — Queue consumer until sentinel

Use a `deque` as a queue. Consume messages until a special sentinel object appears.

The sentinel itself should not be returned.


In [18]:
STOP = object()

queue = deque(["job-1", "job-2", "job-3", STOP, "job-4"])
processed = []

for job in iter(queue.popleft, STOP):
    processed.append(job.upper())

assert processed == ["JOB-1", "JOB-2", "JOB-3"]
assert list(queue) == ["job-4"]


## Problem 16 — Simulate a polling API

Build a polling callable that returns statuses from a queue.

Stop when the status is `"done"`.

Then collect all intermediate statuses.


In [19]:
statuses = deque(["queued", "running", "running", "running", "done", "done"])

def poll_status():
    return statuses.popleft()

history = list(iter(poll_status, "done"))

assert history == ["queued", "running", "running", "running"]
assert list(statuses) == ["done"]


## Problem 17 — Paginate through an API until `None`

Many APIs return a page of data and a cursor for the next page.

Here, a callable hides the cursor state and returns the next page. It returns `None` when no pages remain.


In [20]:
PAGES = {
    None: {"items": [1, 2], "next_cursor": "A"},
    "A": {"items": [3, 4], "next_cursor": "B"},
    "B": {"items": [5], "next_cursor": None},
}


def make_page_fetcher(pages):
    cursor = None
    finished = False

    def fetch_next_page():
        nonlocal cursor, finished

        if finished:
            return None

        page = pages[cursor]
        cursor = page["next_cursor"]

        if cursor is None:
            finished = True

        return page["items"]

    return fetch_next_page


fetch_page = make_page_fetcher(PAGES)
pages = list(iter(fetch_page, None))
items = [item for page in pages for item in page]

assert pages == [[1, 2], [3, 4], [5]]
assert items == [1, 2, 3, 4, 5]


## Problem 18 — Batch processing until an empty batch

When an API returns an empty list to signal completion, use `[]` as the sentinel.

This is intentional equality matching.


In [21]:
batches = deque([[10, 20], [30], [40, 50], []])

def fetch_batch():
    return batches.popleft()

processed = []
for batch in iter(fetch_batch, []):
    processed.extend(x * 2 for x in batch)

assert processed == [20, 40, 60, 80, 100]


## Problem 19 — Mutable sentinel caution

Mutable sentinels can be dangerous if they are modified after the iterator is created.

This example shows why immutable or unique sentinels are usually safer.


In [22]:
sentinel = []
values = deque([[1], [2], []])

it = iter(values.popleft, sentinel)

# Mutating the sentinel changes what future equality comparisons mean.
sentinel.append("changed")

# The deque still contains [], but [] != ["changed"], so the iterator would not stop there.
# We use a guard to avoid an infinite loop after the queue empties.
safe_next = with_max_calls(lambda: next(it), max_calls=3, sentinel="FORCED_STOP")
result = list(iter(safe_next, "FORCED_STOP"))

assert result == [[1], [2], []]


## Problem 20 — Build a peekable callable iterator

Implement an iterator that supports:

- `peek()` to view the next value without consuming it.
- Normal iteration.
- Correct exhaustion behavior.

The sentinel should still not be yielded.


In [23]:
class PeekableCallableIterator:
    def __init__(self, func, sentinel):
        if not callable(func):
            raise TypeError("func must be callable")
        self._func = func
        self._sentinel = sentinel
        self._buffer = deque()
        self._exhausted = False

    def __iter__(self):
        return self

    def _read_next(self):
        if self._exhausted:
            raise StopIteration

        value = self._func()
        if value == self._sentinel:
            self._exhausted = True
            raise StopIteration

        return value

    def peek(self):
        if not self._buffer:
            self._buffer.append(self._read_next())
        return self._buffer[0]

    def __next__(self):
        if self._buffer:
            return self._buffer.popleft()
        return self._read_next()


counter = make_counter()
it = PeekableCallableIterator(counter, 4)

assert it.peek() == 1
assert it.peek() == 1
assert next(it) == 1
assert next(it) == 2
assert it.peek() == 3
assert next(it) == 3

try:
    next(it)
except StopIteration:
    pass
else:
    raise AssertionError("Expected StopIteration")


## Problem 21 — Build a reusable `take_until` helper

Write a function that accepts a zero-argument callable and a sentinel, and returns a list of values produced before the sentinel.

Also support an optional `max_calls` guard.


In [24]:
def take_until(func, sentinel, max_calls=None):
    if max_calls is None:
        return list(iter(func, sentinel))

    guarded = with_max_calls(func, max_calls=max_calls, sentinel=sentinel)
    return list(iter(guarded, sentinel))


counter = make_counter()
assert take_until(counter, 5) == [1, 2, 3, 4]

constant = lambda: "never done"
assert take_until(constant, "done", max_calls=3) == ["never done", "never done", "never done"]


## Problem 22 — Convert a callable iterator into a generator

Implement `iter_call_until(func, sentinel)` as a generator.

Compare it to built-in `iter(func, sentinel)`.


In [25]:
def iter_call_until(func, sentinel):
    while True:
        value = func()
        if value == sentinel:
            return
        yield value


counter_a = make_counter()
counter_b = make_counter()

assert list(iter_call_until(counter_a, 5)) == [1, 2, 3, 4]
assert list(iter(counter_b, 5)) == [1, 2, 3, 4]


## Problem 23 — Compare `iter(callable, sentinel)` with `takewhile`

You can imitate sentinel-based iteration with `itertools.takewhile`, but it is less direct.

`iter(callable, sentinel)` is the clearer tool when you have exactly this pattern.


In [26]:
def repeated_calls(func):
    for _ in count():
        yield func()


counter_a = make_counter()
counter_b = make_counter()

sentinel_iter_values = list(iter(counter_a, 5))
takewhile_values = list(takewhile(lambda value: value != 5, repeated_calls(counter_b)))

assert sentinel_iter_values == [1, 2, 3, 4]
assert takewhile_values == [1, 2, 3, 4]


## Problem 24 — Simulated command input until `"quit"`

Do not use real `input()` in a notebook exercise because it blocks execution.

Instead, simulate user input with a queue.


In [27]:
commands = deque(["help", "status", "run backup", "quit", "delete everything"])

def fake_input():
    return commands.popleft()

accepted_commands = list(iter(fake_input, "quit"))

assert accepted_commands == ["help", "status", "run backup"]
assert list(commands) == ["delete everything"]


## Problem 25 — Streaming transformation pipeline

Read chunks from a stream until empty string, normalize each chunk, and collect non-empty transformed values.

This combines sentinel iteration with a normal processing pipeline.


In [28]:
stream = StringIO("  alpha  \n\n beta\n gamma  ")

def read_line():
    return stream.readline()

cleaned = [
    line.strip().upper()
    for line in iter(read_line, "")
    if line.strip()
]

assert cleaned == ["ALPHA", "BETA", "GAMMA"]


## Problem 26 — Build a robust line reader with line numbers

Create a callable that returns `(line_number, line_text)` until the stream is exhausted.

Since valid data is a tuple, use a unique object as the sentinel.


In [29]:
END = object()

def make_numbered_line_reader(text):
    stream = StringIO(text)
    line_number = 0

    def read_numbered_line():
        nonlocal line_number
        line = stream.readline()
        if line == "":
            return END
        line_number += 1
        return line_number, line.rstrip("\n")

    return read_numbered_line


reader = make_numbered_line_reader("red\nblue\ngreen\n")
lines = list(iter(reader, END))

assert lines == [(1, "red"), (2, "blue"), (3, "green")]


## Problem 27 — Detect whether a callable iterator made progress

Create a function that consumes a callable iterator and returns both:

- the produced values
- a boolean indicating whether the iterator produced at least one value


In [30]:
def collect_with_progress(func, sentinel):
    values = list(iter(func, sentinel))
    return values, bool(values)


counter = make_counter()
values, made_progress = collect_with_progress(counter, 1)

assert values == []
assert made_progress is False

counter = make_counter()
values, made_progress = collect_with_progress(counter, 4)

assert values == [1, 2, 3]
assert made_progress is True


## Problem 28 — Sentinel collision bug

This example intentionally shows a bad sentinel choice.

The data contains `0`, but `0` is also used as the sentinel, so valid data is lost.


In [31]:
data = deque([5, 4, 0, 3, 2])

bad_values = list(iter(data.popleft, 0))

assert bad_values == [5, 4]
assert list(data) == [3, 2]


## Problem 29 — Fix the sentinel collision bug

Use a unique sentinel object that cannot be confused with real numeric data.


In [32]:
END = object()
data = deque([5, 4, 0, 3, 2, END])

good_values = list(iter(data.popleft, END))

assert good_values == [5, 4, 0, 3, 2]


## Problem 30 — Class-based callable object

Closures are not the only way to make stateful callables.

Implement a class with `__call__`.


In [33]:
class ArithmeticProgressionCallable:
    def __init__(self, start=0, step=1):
        self.current = start
        self.step = step

    def __call__(self):
        self.current += self.step
        return self.current


ap = ArithmeticProgressionCallable(start=0, step=3)
values = list(iter(ap, 12))

assert values == [3, 6, 9]


## Problem 31 — Callable object with reset

Create a callable counter object with a `reset()` method.

Use it twice with `iter(callable, sentinel)`.


In [34]:
class ResettableCounter:
    def __init__(self, start=0):
        self.start = start
        self.current = start

    def __call__(self):
        self.current += 1
        return self.current

    def reset(self):
        self.current = self.start


counter = ResettableCounter(start=0)

assert list(iter(counter, 4)) == [1, 2, 3]

# Reusing the same callable without reset is dangerous here:
# it has already passed the sentinel, so it may never return 4 again.
guarded_counter = with_max_calls(counter, max_calls=3, sentinel="FORCED_STOP")
assert list(iter(guarded_counter, "FORCED_STOP")) == [5, 6, 7]

counter.reset()
assert list(iter(counter, 4)) == [1, 2, 3]


## Problem 32 — Avoiding an infinite loop after passing the sentinel

A callable can pass the sentinel and never return it again.

Build `safe_iter_values` that always uses a maximum-call guard.


In [35]:
def safe_iter_values(func, sentinel, max_calls):
    guarded = with_max_calls(func, max_calls=max_calls, sentinel=sentinel)
    return list(iter(guarded, sentinel))


counter = ResettableCounter(start=10)

# This counter starts at 10 and counts up, so it will never return 4.
values = safe_iter_values(counter, sentinel=4, max_calls=5)

assert values == [11, 12, 13, 14, 15]


## Problem 33 — Chunked checksum until end of stream

Read binary chunks until `b""`, calculate a simple checksum, and count the chunks.


In [36]:
reader = BytesReader(b"abcde", block_size=2)

checksum = 0
chunk_count = 0

for block in iter(reader.read, b""):
    checksum += sum(block)
    chunk_count += 1

assert chunk_count == 3
assert checksum == sum(b"abcde")


## Problem 34 — Mini project: retrying a flaky callable until success

A callable returns `"retry"` until it eventually returns `"success"`.

Use sentinel iteration to collect all retries, then prove the success marker was consumed but not yielded.


In [37]:
def make_flaky_operation(failures_before_success):
    attempts = 0

    def run():
        nonlocal attempts
        attempts += 1
        if attempts <= failures_before_success:
            return "retry"
        return "success"

    return run


operation = make_flaky_operation(failures_before_success=3)
retries = list(iter(operation, "success"))

assert retries == ["retry", "retry", "retry"]
assert "success" not in retries


## Problem 35 — Mini project: token stream parser

Consume tokens until an `"EOF"` token appears.

Then transform the token stream into a simple expression string.


In [38]:
tokens = deque(["NUMBER:10", "PLUS", "NUMBER:20", "PLUS", "NUMBER:5", "EOF"])

def next_token():
    return tokens.popleft()

token_values = list(iter(next_token, "EOF"))

expression = (
    " ".join(
        token.split(":", 1)[1] if token.startswith("NUMBER:") else "+"
        for token in token_values
    )
)

assert token_values == ["NUMBER:10", "PLUS", "NUMBER:20", "PLUS", "NUMBER:5"]
assert expression == "10 + 20 + 5"


## Problem 36 — Testing helper: assert a callable iterator equals expected values

Create a reusable assertion helper for future exercises.


In [39]:
def assert_iter_call_equals(func, sentinel, expected):
    actual = list(iter(func, sentinel))
    assert actual == expected, f"Expected {expected}, got {actual}"


assert_iter_call_equals(make_counter(), 4, [1, 2, 3])
assert_iter_call_equals(make_countdown(3), -1, [2, 1, 0])


## Problem 37 — Design question: when not to use `iter(callable, sentinel)`

Sometimes a generator is clearer.

Use a generator when:

- The stop condition is complex.
- You need `try/finally` cleanup around the whole iteration.
- You need to yield multiple values per callable result.
- You need to catch and handle several exception types differently.
- The callable requires changing arguments on every call.

The next solution shows a generator that expands each batch into individual items.


In [40]:
def iter_items_from_batches(fetch_batch):
    while True:
        batch = fetch_batch()
        if batch == []:
            return
        for item in batch:
            yield item


batches = deque([[1, 2], [3], [4, 5], []])

assert list(iter_items_from_batches(batches.popleft)) == [1, 2, 3, 4, 5]


## Problem 38 — Final challenge: build a monitored callable iterator

Create a wrapper that records:

- how many calls were made
- how many values were yielded
- whether the sentinel was reached

Return the iterator and a metrics dictionary.


In [41]:
def monitored_iter(func, sentinel):
    metrics = {
        "calls": 0,
        "yielded": 0,
        "sentinel_reached": False,
    }

    def monitored_func():
        value = func()
        metrics["calls"] += 1

        if value == sentinel:
            metrics["sentinel_reached"] = True
        else:
            metrics["yielded"] += 1

        return value

    return iter(monitored_func, sentinel), metrics


counter = make_counter()
it, metrics = monitored_iter(counter, 5)

assert list(it) == [1, 2, 3, 4]
assert metrics == {
    "calls": 5,
    "yielded": 4,
    "sentinel_reached": True,
}


## Summary

Key ideas:

- `iter(callable, sentinel)` repeatedly calls a zero-argument callable.
- Iteration stops when the returned value equals the sentinel.
- The sentinel is consumed but not yielded.
- Sentinel matching uses equality, so choose sentinels carefully.
- Use unique sentinel objects when `None`, `0`, `""`, or `[]` may be valid data.
- Add guardrails when there is a chance the callable will never return the sentinel.
- For simple "call until value" workflows, this pattern is clean and Pythonic.
